## Setup

Keep `example_data.py` and `utility_metrics.py` beside this notebook. Scikit-learn is supplied by the project environment.

In [1]:
#| label: setup
from IPython.display import display

from example_data import TARGET_COLUMN, make_example_views
from utility_metrics import (
    holdout_auroc_details,
    independent_holdout_auroc_difference,
)

## 1. Metric and inputs

| Property | Value |
|---|---|
| Metric | Independent-holdout AUROC difference |
| Family | Utility |
| Origin | Established and adapted in the source project |
| Classifier | Predicts the task outcome |
| Input | $\widetilde X,\widetilde Y$ |
| Holdout | Independent $\widetilde H$ |
| Range | $[-1,1]$; zero means equal average utility |

Models trained on $X$ and $Y$ are never evaluated on their training rows. Both sides use the same independent holdout, so their AUROCs are directly comparable.

## 2. AUROC

For a scoring model $f$ and one randomly selected positive and negative holdout row, AUROC has the ranking interpretation

$$
\operatorname{AUC}(f;H)=
P(f(h^+)>f(h^-))+\frac12P(f(h^+)=f(h^-)).
$$

It is threshold-free. A value of $0.5$ indicates chance ranking and $1$ indicates perfect ranking.

## 3. Average over the same model families

For each model family $m=1,\ldots,K$, fit

$$
f_m^X\text{ on }\widetilde X,\qquad f_m^Y\text{ on }\widetilde Y,
$$

and evaluate both on $\widetilde H$. Define

$$
\bar A_X=\frac1K\sum_{m=1}^{K}\operatorname{AUC}(f_m^X;\widetilde H),
\qquad
\bar A_Y=\frac1K\sum_{m=1}^{K}\operatorname{AUC}(f_m^Y;\widetilde H).
$$

The reported directional difference is

$$
D_{\mathrm{AUROC}}(X,Y;H)=\bar A_Y-\bar A_X.
$$

The source implementation uses $K=4$: random forest, AdaBoost, RBF support-vector classifier, and logistic regression. A constant training target uses a dummy classifier and receives AUROC $0.5$.

In [2]:
#| label: holdout-auroc

example = make_example_views()
X_tilde, Y_tilde, H_tilde = example.X_tilde, example.Y_tilde, example.H_tilde

details = holdout_auroc_details(
    reference_data=X_tilde,
    comparison_data=Y_tilde,
    holdout_data=H_tilde,
    target_column=TARGET_COLUMN,
)
D_auroc = independent_holdout_auroc_difference(
    reference_data=X_tilde,
    comparison_data=Y_tilde,
    holdout_data=H_tilde,
    target_column=TARGET_COLUMN,
)

display(details.round(4))
print(f"Average AUROC trained on X = {details['AUROC_X'].mean():.6f}")
print(f"Average AUROC trained on Y = {details['AUROC_Y'].mean():.6f}")
print(f"Independent-holdout AUROC difference = {D_auroc:.6f}")

,model,AUROC_X,AUROC_Y,difference_Y_minus_X
0,Random forest,0.8958,0.8542,-0.0417
1,AdaBoost,1.0000,0.8750,-0.1250
2,RBF SVC,0.6250,0.7083,0.0833
3,Logistic regression,0.6667,0.5833,-0.0833


Average AUROC trained on X = 0.796875
Average AUROC trained on Y = 0.755208
Independent-holdout AUROC difference = -0.041667


## 4. Interpretation

The example gives a small negative difference: models trained on $Y$ rank the shared holdout slightly worse on average than models trained on $X$. Positive values mean $Y$ provides better predictive utility for this task; negative values mean worse utility; zero means equal average AUROC.

This metric is directional, task-specific, and model-set-specific. Swapping $X$ and $Y$ changes the sign. It measures utility rather than general distributional similarity.

## Takeaway

Train the same classifier families independently on $\widetilde X$ and $\widetilde Y$, score every model on the same untouched $\widetilde H$, average within each side, and report $\bar A_Y-\bar A_X$.